In [1]:
"""
Rolling Horizon Production Planning in Pyomo
--------------------------------------------
We have:
- periods t = 1..T
- production decision x[t]
- inventory I[t]
- backorder B[t] (unmet demand carried forward)

Flow balance each period:
I[t] - B[t] = I[t-1] - B[t-1] + x[t] - demand[t]

Rolling horizon:
- choose a planning window H (e.g., 6 weeks)
- at time k, solve for k..k+H-1
- implement ONLY decisions for period k
- update initial inventory/backorder and move to k+1
"""

import pyomo.environ as pyo

def solve_window(demand_window, cap_window, init_I, init_B,
                 prod_cost=2.0, hold_cost=0.4, back_cost=5.0):
    """
    Solve one planning window (a small MILP/LP).
    Returns the optimal first-period decisions and the end-of-period states.
    """

    H = len(demand_window)  # window length
    m = pyo.ConcreteModel()

    # Sets
    m.T = pyo.RangeSet(1, H)

    # Parameters (indexed by t in 1..H)
    m.d = pyo.Param(m.T, initialize={t: demand_window[t-1] for t in range(1, H+1)})
    m.cap = pyo.Param(m.T, initialize={t: cap_window[t-1] for t in range(1, H+1)})

    # Decision variables
    m.x = pyo.Var(m.T, domain=pyo.NonNegativeReals)  # production
    m.I = pyo.Var(m.T, domain=pyo.NonNegativeReals)  # inventory
    m.B = pyo.Var(m.T, domain=pyo.NonNegativeReals)  # backorder

    # Capacity constraint
    def cap_rule(m, t):
        return m.x[t] <= m.cap[t]
    m.Capacity = pyo.Constraint(m.T, rule=cap_rule)

    # Inventory/backorder balance:
    # I[t] - B[t] = (I[t-1] - B[t-1]) + x[t] - d[t]
    def flow_rule(m, t):
        if t == 1:
            return m.I[t] - m.B[t] == (init_I - init_B) + m.x[t] - m.d[t]
        return m.I[t] - m.B[t] == (m.I[t-1] - m.B[t-1]) + m.x[t] - m.d[t]
    m.Flow = pyo.Constraint(m.T, rule=flow_rule)

    # Objective: production + holding + backorder penalties
    m.Obj = pyo.Objective(
        expr=sum(prod_cost*m.x[t] + hold_cost*m.I[t] + back_cost*m.B[t] for t in m.T),
        sense=pyo.minimize
    )

    # Solve
    solver = pyo.SolverFactory("glpk")
    res = solver.solve(m, tee=False)

    # Extract first period decisions and states after period 1
    x1 = pyo.value(m.x[1])
    I1 = pyo.value(m.I[1])
    B1 = pyo.value(m.B[1])

    # Also extract full plan for debugging/inspection
    plan = {
        "x": [pyo.value(m.x[t]) for t in m.T],
        "I": [pyo.value(m.I[t]) for t in m.T],
        "B": [pyo.value(m.B[t]) for t in m.T],
        "obj": pyo.value(m.Obj)
    }

    return x1, I1, B1, plan


def rolling_horizon_run(demand, capacity, H=6, init_I=10.0, init_B=0.0):
    """
    Run rolling horizon across the whole timeline.
    - demand, capacity: lists of length T_total
    - H: planning window size
    """

    T_total = len(demand)
    assert len(capacity) == T_total

    # Storage for executed decisions
    executed_x = []
    executed_I = []
    executed_B = []

    I_state = init_I
    B_state = init_B

    for k in range(T_total):
        # Build window data from k to k+H-1
        # If near the end, we shorten the window.
        end = min(k + H, T_total)
        demand_window = demand[k:end]
        cap_window = capacity[k:end]

        x1, I1, B1, plan = solve_window(
            demand_window=demand_window,
            cap_window=cap_window,
            init_I=I_state,
            init_B=B_state,
            prod_cost=2.0,
            hold_cost=0.4,
            back_cost=5.0
        )

        # Implement ONLY first period
        executed_x.append(x1)
        executed_I.append(I1)
        executed_B.append(B1)

        # Update system state for next step (this is the key rolling part)
        I_state = I1
        B_state = B1

        print(f"Period {k+1:02d} | demand={demand[k]:>5.1f} cap={capacity[k]:>5.1f} "
              f"-> produce={x1:>6.1f}  end_I={I1:>6.1f}  end_B={B1:>6.1f}")

    return executed_x, executed_I, executed_B


if __name__ == "__main__":
    # Example data (12 periods)
    demand =   [12, 15, 18, 10,  9, 20, 25, 22, 14, 13, 19, 16]
    capacity = [14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14]

    executed_x, executed_I, executed_B = rolling_horizon_run(
        demand=demand,
        capacity=capacity,
        H=6,        # plan 6 periods ahead each time
        init_I=10,  # starting inventory
        init_B=0    # starting backorder
    )

    print("\nExecuted production:", [round(v, 1) for v in executed_x])
    print("Ending inventory:    ", [round(v, 1) for v in executed_I])
    print("Ending backorders:   ", [round(v, 1) for v in executed_B])


Period 01 | demand= 12.0 cap= 14.0 -> produce=   7.0  end_I=   5.0  end_B=   0.0
Period 02 | demand= 15.0 cap= 14.0 -> produce=  14.0  end_I=   4.0  end_B=   0.0
Period 03 | demand= 18.0 cap= 14.0 -> produce=  14.0  end_I=   0.0  end_B=   0.0
Period 04 | demand= 10.0 cap= 14.0 -> produce=  14.0  end_I=   4.0  end_B=   0.0
Period 05 | demand=  9.0 cap= 14.0 -> produce=  14.0  end_I=   9.0  end_B=   0.0
Period 06 | demand= 20.0 cap= 14.0 -> produce=  14.0  end_I=   3.0  end_B=   0.0
Period 07 | demand= 25.0 cap= 14.0 -> produce=  14.0  end_I=   0.0  end_B=   8.0
Period 08 | demand= 22.0 cap= 14.0 -> produce=  14.0  end_I=   0.0  end_B=  16.0
Period 09 | demand= 14.0 cap= 14.0 -> produce=  14.0  end_I=   0.0  end_B=  16.0
Period 10 | demand= 13.0 cap= 14.0 -> produce=  14.0  end_I=   0.0  end_B=  15.0
Period 11 | demand= 19.0 cap= 14.0 -> produce=  14.0  end_I=   0.0  end_B=  20.0
Period 12 | demand= 16.0 cap= 14.0 -> produce=  14.0  end_I=   0.0  end_B=  22.0

Executed production: [7.0, 